# 🚀 Unidad 3 — Clase 5: Quicksort

## Información del Curso

| Aspecto | Detalle |
|--------|--------|
| **Universidad** | Universidad de Talca, Chile |
| **Carrera** | Ingeniería Civil en Informática |
| **Semestre** | 2°-3° año |
| **Curso** | Algoritmos y Estructuras de Datos |
| **Docente** | PhD. César Astudillo |
| **Clase** | Unidad 3, Clase 5 — Quicksort |
| **Duración** | 50 minutos |

---
> 🎯 *Este notebook está diseñado para ser ejecutado en clase de forma interactiva.  
> Ejecuta las celdas en orden de arriba hacia abajo.*

## Verificación de Dependencias

In [1]:
import random
import time
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches ## Para crear leyendas personalizadas
import numpy as np
from IPython.display import display, HTML ## Para mostrar tablas de manera más atractiva
print("✅ Dependencias cargadas correctamente")

✅ Dependencias cargadas correctamente


## 🎯 Objetivos de Aprendizaje

Al finalizar esta sesión, el estudiante será capaz de:

1. **Explicar** el concepto de partición como núcleo de Quicksort.
2. **Implementar** Quicksort con partición de Lomuto y de Hoare.
3. **Analizar** el impacto del pivot en la complejidad: O(n log n) promedio vs O(n²) peor caso.
4. **Comparar** distintas estrategias de selección de pivot (primero, último, aleatorio, mediana de tres).
5. **Contrastar** Quicksort con Merge Sort en términos de complejidad, memoria y rendimiento práctico.

# Sección 1: ¿Por qué Quicksort después de Merge Sort? (5 minutos)

## Recap: lo que sabemos

En la clase anterior vimos **Merge Sort**:
- Paradigma: Divide & Conquer
- Complejidad: **O(n log n) siempre** (mejor, promedio y peor caso)
- Memoria extra: **O(n)** — necesita arreglo auxiliar

Entonces... ¿para qué aprender otro algoritmo O(n log n)?

> 🎙️ **[PAUSA PROFESOR]** *"¿Alguien tiene una hipótesis de por qué Quicksort podría ser mejor que Merge Sort en la práctica, si tienen la misma complejidad asintótica?"*

## La respuesta: las constantes importan

La notación O(n log n) **oculta factores constantes**:

| Algoritmo | Ops/elemento aprox. | Memoria extra | Cache-friendly |
|-----------|---------------------|---------------|----------------|
| Merge Sort | ~2n log n | O(n) | Moderado |
| Quicksort | ~1.4n log n (promedio) | O(log n) pila | **Sí** |

Quicksort es **in-place** y tiene mejor localidad de caché → en la práctica es ~2× más rápido que Merge Sort para datos en memoria RAM.

> 📌 Por eso `Arrays.sort()` en Java, `qsort()` en C, y variantes de Quicksort son la elección por defecto en la mayoría de las librerías estándar.

# Sección 2: La Idea Central — La Partición (10 minutos)

## El insight fundamental

Quicksort se basa en una observación simple:

> Si encontramos un elemento que ya está **en su posición final correcta**,  
> podemos ordenar el resto de forma independiente.

¿Cuándo un elemento está en su posición final?  
**Cuando todos los elementos menores están a su izquierda y todos los mayores a su derecha.**

A ese elemento lo llamamos **pivot**.

## ¿Qué es una partición?

Dado un arreglo y un pivot elegido:
```
ANTES:  [3, 6, 8, 10, 1, 2, 1]   pivot = 3

DESPUÉS: [1, 2, 1,  3,  6, 8, 10]
          ↑        ↑   ↑
       menores   pivot  mayores
```

La función `particionar(arr, inicio, fin)` debe:
1. Elegir un pivot
2. Reorganizar los elementos: menores antes, mayores después
3. **Devolver el índice final del pivot**

> 🎙️ **[PAUSA PROFESOR]** *"La clave es que el pivot queda en su posición definitiva. ¿Cuántas veces se moverá el pivot después de la partición?"*  
> *Respuesta esperada: nunca más, está en su lugar final.*

## La recursión de Quicksort

```
quicksort(arr, inicio, fin):
    si inicio < fin:
        p = particionar(arr, inicio, fin)  # p es el índice del pivot
        quicksort(arr, inicio, p - 1)      # ordenar la mitad izquierda
        quicksort(arr, p + 1, fin)         # ordenar la mitad derecha
```

La elegancia: **no hay paso "Combine"** — todo el trabajo es en la partición.

# Sección 3: Partición de Lomuto (12 minutos)

## 3.1 El Esquema de Lomuto

La partición de Lomuto elige **el último elemento** como pivot y usa **un solo puntero** `i`.

### Invariante del algoritmo

En todo momento:
- `arr[inicio..i]` contiene elementos **≤ pivot**
- `arr[i+1..j-1]` contiene elementos **> pivot**
- `arr[j..fin-1]` son los elementos aún no procesados

```
arr = [3, 8, 5, 1, 2, 6, 7, 4]   pivot = 4 (último)
       i=-1 (comienza antes de inicio)

j=0: arr[0]=3 ≤ 4 → i=0, swap(arr[0],arr[0]) → [3, 8, 5, 1, 2, 6, 7, 4]
j=1: arr[1]=8 > 4 → no swap
j=2: arr[2]=5 > 4 → no swap
j=3: arr[3]=1 ≤ 4 → i=1, swap(arr[1],arr[3]) → [3, 1, 5, 8, 2, 6, 7, 4]
j=4: arr[4]=2 ≤ 4 → i=2, swap(arr[2],arr[4]) → [3, 1, 2, 8, 5, 6, 7, 4]
j=5: arr[5]=6 > 4 → no swap
j=6: arr[6]=7 > 4 → no swap
Fin: swap(arr[i+1], arr[fin]) = swap(arr[3], arr[7]) → [3, 1, 2, 4, 5, 6, 7, 8]
                                                                    ↑
                                                              pivot en pos 3
```

In [2]:
def particion_lomuto(arr, inicio, fin):
    """
    Partición de Lomuto: pivot = arr[fin]
    Retorna el índice final del pivot.
    Cuenta intercambios y comparaciones para análisis.
    """
    pivot = arr[fin] ## Elegimos el último elemento como pivot
    i = inicio - 1  # índice del último elemento ≤ pivot
    
    comparaciones = 0
    intercambios = 0
    
    for j in range(inicio, fin): ## Recorremos desde el inicio hasta el penúltimo elemento
        comparaciones += 1
        if arr[j] <= pivot: ## Si el elemento actual es menor o igual al pivot
            i += 1 # Incrementamos el índice del último elemento ≤ pivot
            arr[i], arr[j] = arr[j], arr[i] ## Intercambiamos el elemento actual con el elemento en la posición i
            intercambios += 1
    
    # Poner el pivot en su posición final
    arr[i + 1], arr[fin] = arr[fin], arr[i + 1]
    intercambios += 1
    
    return i + 1, comparaciones, intercambios


def quicksort_lomuto(arr, inicio=None, fin=None, stats=None):
    """Quicksort con partición de Lomuto."""
    if inicio is None:
        inicio = 0
    if fin is None:
        fin = len(arr) - 1
    if stats is None:
        stats = {'comparaciones': 0, 'intercambios': 0}
    
    if inicio < fin:
        idx_pivot, comps, swaps = particion_lomuto(arr, inicio, fin)
        stats['comparaciones'] += comps
        stats['intercambios'] += swaps
        quicksort_lomuto(arr, inicio, idx_pivot - 1, stats)
        quicksort_lomuto(arr, idx_pivot + 1, fin, stats)
    
    return stats

# Demostración
ejemplo = [3, 8, 5, 1, 2, 6, 7, 4]
print(f"Antes:   {ejemplo}")
stats = quicksort_lomuto(ejemplo)
print(f"Después: {ejemplo}")
print(f"Comparaciones: {stats['comparaciones']}, Intercambios: {stats['intercambios']}")

Antes:   [3, 8, 5, 1, 2, 6, 7, 4]
Después: [1, 2, 3, 4, 5, 6, 7, 8]
Comparaciones: 15, Intercambios: 15


## 3.2 Trazando Lomuto paso a paso

> 🎙️ **[PAUSA PROFESOR]** *"Ejecutemos con [5, 3, 8, 1, 7] y tracemos cada paso juntos en la pizarra antes de ver el resultado."*

In [3]:
def particion_lomuto_verbose(arr, inicio, fin):
    """Versión pedagógica que muestra cada paso."""
    arr = arr.copy()
    pivot = arr[fin]
    print(f"  pivot = {pivot} (posición {fin})")
    i = inicio - 1
    
    for j in range(inicio, fin):
        if arr[j] <= pivot:
            i += 1
            arr[i], arr[j] = arr[j], arr[i]
            print(f"  arr[{j}]={arr[i]} ≤ pivot → swap pos {i}↔{j}: {arr}")
        else:
            print(f"  arr[{j}]={arr[j]} > pivot  → sin swap: {arr}")
    
    arr[i + 1], arr[fin] = arr[fin], arr[i + 1]
    print(f"  Pivot al centro: swap pos {i+1}↔{fin}: {arr}")
    print(f"  → Pivot {pivot} queda en índice {i+1}")
    return i + 1

ejemplo = [5, 3, 8, 1, 7]
print(f"Arreglo inicial: {ejemplo}")
idx = particion_lomuto_verbose(ejemplo, 0, len(ejemplo)-1)
print(f"\n✅ Pivote en índice {idx}: {ejemplo}")

Arreglo inicial: [5, 3, 8, 1, 7]
  pivot = 7 (posición 4)
  arr[0]=5 ≤ pivot → swap pos 0↔0: [5, 3, 8, 1, 7]
  arr[1]=3 ≤ pivot → swap pos 1↔1: [5, 3, 8, 1, 7]
  arr[2]=8 > pivot  → sin swap: [5, 3, 8, 1, 7]
  arr[3]=1 ≤ pivot → swap pos 2↔3: [5, 3, 1, 8, 7]
  Pivot al centro: swap pos 3↔4: [5, 3, 1, 7, 8]
  → Pivot 7 queda en índice 3

✅ Pivote en índice 3: [5, 3, 8, 1, 7]


# Sección 4: Partición de Hoare (10 minutos)

## 4.1 La versión original (y más eficiente)

Tony Hoare diseñó Quicksort en 1959. Su esquema de partición usa **dos punteros** que se mueven desde los extremos hacia el centro.

### La idea

```
arr = [3, 5, 7, 1, 2, 4, 6, 8]   pivot = arr[0] = 3

i →→→→→→→→              ←←←←←←←← j
[3,  5,  7,  1,  2,  4,  6,  8]
 i=0                          j=7

i busca: arr[i] < pivot (avanza →)
j busca: arr[j] > pivot (avanza ←)

→ cuando ambos se detienen, swap(arr[i], arr[j])
→ repetir hasta que i >= j

Iteración 1:
  i: arr[0]=3, 3 < 3? No → i se detiene en 0
  j: arr[7]=8 > 3 → arr[6]=6 > 3 → arr[5]=4 > 3 → arr[4]=2, 2 > 3? No → j se detiene en 4
  i(0) < j(4) → swap(arr[0], arr[4]):  [2, 5, 7, 1, 3, 4, 6, 8]

Iteración 2:
  i: arr[1]=5, 5 < 3? No → i se detiene en 1
  j: arr[3]=1, 1 > 3? No → j se detiene en 3
  i(1) < j(3) → swap(arr[1], arr[3]):  [2, 1, 7, 5, 3, 4, 6, 8]

Iteración 3:
  i: arr[2]=7, 7 < 3? No → i se detiene en 2
  j: arr[2]=7 > 3 → arr[1]=1, 1 > 3? No → j se detiene en 1
  i(2) ≥ j(1) → retorna j=1  ← índice de partición

Resultado: [2, 1, | 7, 5, 3, 4, 6, 8]
                  ↑
            partición en índice 1
  (todo en [0..1] ≤ 3,  todo en [2..7] ≥ 3)
```

**Diferencia clave con Lomuto:**
- Hoare hace ~3× menos intercambios en promedio
- El pivot **no** queda necesariamente en `j` al final — puede estar en cualquier parte
- Retorna el índice de partición (no el índice del pivot)

In [ ]:
def particion_hoare(arr, inicio, fin):
    """
    Partición de Hoare: pivot = arr[inicio]
    Retorna índice de partición (no necesariamente la posición del pivot).
    """
    pivot = arr[inicio]
    i = inicio - 1
    j = fin + 1
    
    comparaciones = 0
    intercambios = 0
    
    while True:
        # Mover i hacia la derecha mientras arr[i] < pivot
        i += 1
        while arr[i] < pivot:
            comparaciones += 1
            i += 1
        comparaciones += 1  # la comparación que falló
        
        # Mover j hacia la izquierda mientras arr[j] > pivot
        j -= 1
        while arr[j] > pivot:
            comparaciones += 1
            j -= 1
        comparaciones += 1  # la comparación que falló
        
        if i >= j:
            return j, comparaciones, intercambios
        
        arr[i], arr[j] = arr[j], arr[i]
        intercambios += 1


def quicksort_hoare(arr, inicio=None, fin=None, stats=None):
    """Quicksort con partición de Hoare."""
    if inicio is None:
        inicio = 0
    if fin is None:
        fin = len(arr) - 1
    if stats is None:
        stats = {'comparaciones': 0, 'intercambios': 0}
    
    if inicio < fin:
        p, comps, swaps = particion_hoare(arr, inicio, fin)
        stats['comparaciones'] += comps
        stats['intercambios'] += swaps
        quicksort_hoare(arr, inicio, p, stats)
        quicksort_hoare(arr, p + 1, fin, stats)
    
    return stats

# Comparación Lomuto vs Hoare
import random
datos = random.sample(range(1000), 200)

arr1 = datos.copy()
arr2 = datos.copy()

stats_lomuto = quicksort_lomuto(arr1)
stats_hoare = quicksort_hoare(arr2)

print("Comparación sobre 200 elementos aleatorios:")
print(f"{'Métrica':<20} {'Lomuto':>12} {'Hoare':>12} {'Ratio':>10}")
print("-" * 55)
print(f"{'Comparaciones':<20} {stats_lomuto['comparaciones']:>12} {stats_hoare['comparaciones']:>12} {stats_lomuto['comparaciones']/stats_hoare['comparaciones']:>10.2f}x")
print(f"{'Intercambios':<20} {stats_lomuto['intercambios']:>12} {stats_hoare['intercambios']:>12} {stats_lomuto['intercambios']/max(stats_hoare['intercambios'],1):>10.2f}x")

assert arr1 == sorted(datos) and arr2 == sorted(datos), "❌ Error en ordenamiento"
print("\n✅ Ambas implementaciones correctas")

# Sección 5: Estrategias de Pivot (8 minutos)

## ¿Por qué importa el pivot?

La elección del pivot determina el **balance de la partición**:

- **Pivot ideal:** divide en dos mitades iguales → árbol equilibrado → O(n log n)
- **Peor pivot:** siempre elige el mínimo o máximo → árbol degenerado → **O(n²)**

## Las 4 estrategias principales

In [ ]:
def pivot_primer_elemento(arr, inicio, fin):
    """Pivot = arr[inicio]. Simple pero O(n²) en arreglos ya ordenados.""""
    return arr[inicio]

def pivot_ultimo_elemento(arr, inicio, fin):
    """Pivot = arr[fin]. Igual que Lomuto original.""""
    return arr[fin]

def pivot_aleatorio(arr, inicio, fin):
    """Pivot aleatorio. Elimina el peor caso determinista.""""
    idx = random.randint(inicio, fin)
    arr[inicio], arr[idx] = arr[idx], arr[inicio]  # mover al inicio para Hoare
    return arr[inicio]

def pivot_mediana_de_tres(arr, inicio, fin):
    """
    Pivot = mediana de arr[inicio], arr[mid], arr[fin].
    Buen equilibrio en la práctica — usado en introsort.
    """
    mid = (inicio + fin) // 2
    # Ordenar los tres candidatos en sus posiciones
    if arr[inicio] > arr[mid]:
        arr[inicio], arr[mid] = arr[mid], arr[inicio]
    if arr[inicio] > arr[fin]:
        arr[inicio], arr[fin] = arr[fin], arr[inicio]
    if arr[mid] > arr[fin]:
        arr[mid], arr[fin] = arr[fin], arr[mid]
    # arr[mid] es la mediana; moverla al inicio
    arr[inicio], arr[mid] = arr[mid], arr[inicio]
    return arr[inicio]

# Demostración: peor caso del pivot ingenuo (arreglo ya ordenado)
n = 500
arr_ordenado = list(range(n))

import sys
sys.setrecursionlimit(10000)

# Con pivot = último elemento sobre arreglo ordenado → O(n²)
def contar_comparaciones_lomuto(arr, inicio, fin, contador):
    if inicio < fin:
        pivot = arr[fin]
        i = inicio - 1
        for j in range(inicio, fin):
            contador[0] += 1
            if arr[j] <= pivot:
                i += 1
                arr[i], arr[j] = arr[j], arr[i]
        arr[i+1], arr[fin] = arr[fin], arr[i+1]
        p = i + 1
        contar_comparaciones_lomuto(arr, inicio, p - 1, contador)
        contar_comparaciones_lomuto(arr, p + 1, fin, contador)

arr1 = list(range(n))
arr2 = list(range(n))

# Pivot último (peor caso para ordenado)
cont1 = [0]
try:
    contar_comparaciones_lomuto(arr1, 0, n-1, cont1)
    comp_peor = cont1[0]
except RecursionError:
    comp_peor = -1

# Con pivot aleatorio (evita el peor caso)
def quicksort_pivot_aleatorio(arr, inicio, fin, contador):
    if inicio < fin:
        idx = random.randint(inicio, fin)
        arr[inicio], arr[idx] = arr[idx], arr[inicio]
        pivot = arr[fin]
        i = inicio - 1
        for j in range(inicio, fin):
            contador[0] += 1
            if arr[j] <= pivot:
                i += 1
                arr[i], arr[j] = arr[j], arr[i]
        arr[i+1], arr[fin] = arr[fin], arr[i+1]
        p = i + 1
        quicksort_pivot_aleatorio(arr, inicio, p-1, contador)
        quicksort_pivot_aleatorio(arr, p+1, fin, contador)

cont2 = [0]
quicksort_pivot_aleatorio(arr2, 0, n-1, cont2)

teorico_nlogn = int(n * np.log2(n))
teorico_n2 = n * n

print(f"Arreglo ya ordenado, n={n}")
print(f"{'Estrategia':<25} {'Comparaciones':>15} {'Teórico':>15}")
print("-" * 56)
if comp_peor > 0:
    print(f"{'Pivot último (peor caso)':<25} {comp_peor:>15,} {teorico_n2:>15,}  ← O(n²)")
else:
    print(f"{'Pivot último (peor caso)':<25} {'RecursionError':>15} {teorico_n2:>15,}  ← O(n²)")
print(f"{'Pivot aleatorio':<25} {cont2[0]:>15,} {teorico_nlogn:>15,}  ← O(n log n)")

## Resumen de estrategias de pivot

| Estrategia | Implementación | Peor caso | Promedio | Usado en |
|-----------|---------------|-----------|----------|---------|
| Primer/Último elemento | Trivial | O(n²) para datos ordenados | O(n log n) | Ejercicios académicos |
| Aleatorio | `random.randint` | O(n²) improbable | O(n log n) | Cuando se desconfía del input |
| Mediana de 3 | 3 comparaciones | O(n²) muy raro | ~10% mejor que aleatorio | C++ `std::sort`, Python |
| Mediana de medianas | O(n) | **O(n log n) garantizado** | O(n log n) | Introsort, análisis teórico |

> 🎙️ **[PAUSA PROFESOR]** *"¿Por qué Python usa Timsort y no Quicksort para `sorted()`? Hint: piensen en estabilidad."*

# Sección 6: Análisis de Complejidad (8 minutos)

## 6.1 Los tres escenarios

### Peor caso — O(n²)
Ocurre cuando la partición es siempre completamente desbalanceada (pivot = mínimo o máximo):

$$T(n) = T(n-1) + T(0) + O(n) = T(n-1) + O(n)$$

Resolviendo: $T(n) = O(n^2)$

**Ejemplo:** Quicksort con pivot=último sobre arreglo **ya ordenado** o **inversamente ordenado**.

### Mejor caso — O(n log n)
Ocurre cuando el pivot siempre divide en dos mitades exactamente iguales:

$$T(n) = 2T(n/2) + O(n)$$

Por el Teorema Maestro (igual que Merge Sort): $T(n) = O(n \log n)$

### Caso promedio — O(n log n)
Para una permutación aleatoria, el pivot esperado divide en partes proporcionales.  
El análisis exacto da:

$$T(n) \approx 2n \ln n \approx 1.386 \cdot n \log_2 n$$

Esto es **~39% más que el caso ideal** pero sigue siendo O(n log n).

## 6.2 Complejidad espacial

| Caso | Espacio de pila |
|------|----------------|
| Mejor/Promedio | O(log n) |
| Peor | O(n) ← ¡posible stack overflow! |

> Con optimización **tail recursion** en la rama mayor, se garantiza O(log n) siempre.

In [ ]:
# Visualización empírica de las tres complejidades
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sizes = [50, 100, 200, 400, 800, 1600]
comps_aleatorio = []
comps_ordenado  = []
comps_teorico   = []

sys.setrecursionlimit(50000)

for n in sizes:
    # Caso promedio: datos aleatorios
    datos = random.sample(range(n * 10), n)
    arr = datos.copy()
    s = quicksort_lomuto(arr)
    comps_aleatorio.append(s['comparaciones'])
    
    # Caso teórico O(n log n)
    comps_teorico.append(int(2 * n * np.log2(max(n, 2))))
    
    # Peor caso: datos ordenados con pivot último
    arr_ord = list(range(n))
    cont = [0]
    try:
        contar_comparaciones_lomuto(arr_ord, 0, n-1, cont)
        comps_ordenado.append(cont[0])
    except RecursionError:
        comps_ordenado.append(n*n//2)  # estimación

# Gráfico izquierdo: comparaciones absolutas
ax1 = axes[0]
ax1.plot(sizes, comps_aleatorio, 'b-o', label='Quicksort datos aleatorios', linewidth=2)
ax1.plot(sizes, comps_teorico,   'g--s', label='O(n log n) teórico', linewidth=2)
ax1.plot(sizes, comps_ordenado,  'r-^', label='Quicksort datos ordenados (peor caso)', linewidth=2)
ax1.set_xlabel('n (tamaño)', fontsize=12)
ax1.set_ylabel('Comparaciones', fontsize=12)
ax1.set_title('Comparaciones vs n', fontsize=13)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Gráfico derecho: normalizado por n log n para ver si converge
ax2 = axes[1]
nlogn = [n * np.log2(max(n,2)) for n in sizes]
ax2.plot(sizes, [c/t for c,t in zip(comps_aleatorio, nlogn)], 'b-o', 
         label='Aleatorio / (n log n)', linewidth=2)
ax2.plot(sizes, [c/(n*n) for c,n in zip(comps_ordenado, sizes)], 'r-^', 
         label='Ordenado / n²', linewidth=2)
ax2.axhline(y=1.386, color='g', linestyle='--', label='~1.386 (promedio teórico)')
ax2.set_xlabel('n (tamaño)', fontsize=12)
ax2.set_ylabel('Ratio normalizado', fontsize=12)
ax2.set_title('Normalización (confirmar O-grande)', fontsize=13)
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 2.5)

plt.suptitle('Análisis de Complejidad de Quicksort', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('quicksort_complejidad.png', dpi=120, bbox_inches='tight')
plt.show()
print("\n💡 El ratio Aleatorio/(n log n) converge a ~1.386: confirma O(n log n) promedio.")
print("💡 El ratio Ordenado/n² converge a ~0.5: confirma O(n²) para datos ordenados.")

# Sección 7: Quicksort vs Merge Sort (5 minutos)

## Comparación teórica

| Criterio | Quicksort | Merge Sort |
|----------|-----------|------------|
| **Peor caso** | O(n²) | O(n log n) ✅ |
| **Caso promedio** | O(n log n) ✅ | O(n log n) ✅ |
| **Memoria extra** | O(log n) ✅ | O(n) |
| **Estable** | No ❌ | Sí ✅ |
| **In-place** | Sí ✅ | No (típicamente) |
| **Cache-friendly** | Sí ✅ | Moderado |
| **Constante práctica** | ~1.4n log n ✅ | ~2n log n |

In [ ]:
# Benchmark empírico: Quicksort vs Merge Sort
def merge_sort(arr):
    """Merge Sort del notebook anterior.""""
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    izq = merge_sort(arr[:mid])
    der = merge_sort(arr[mid:])
    return merge(izq, der)

def merge(izq, der):
    resultado = []
    i = j = 0
    while i < len(izq) and j < len(der):
        if izq[i] <= der[j]:
            resultado.append(izq[i]); i += 1
        else:
            resultado.append(der[j]); j += 1
    resultado.extend(izq[i:])
    resultado.extend(der[j:])
    return resultado

def quicksort_inplace(arr, inicio=0, fin=None):
    if fin is None: fin = len(arr) - 1
    if inicio < fin:
        idx, _, _ = particion_lomuto(arr, inicio, fin)
        quicksort_inplace(arr, inicio, idx - 1)
        quicksort_inplace(arr, idx + 1, fin)

# Benchmark
tamaños = [1000, 5000, 10000, 50000]
resultados = []

for n in tamaños:
    datos = [random.randint(0, n*10) for _ in range(n)]
    
    # Merge Sort
    t0 = time.perf_counter()
    for _ in range(5):
        merge_sort(datos.copy())
    t_merge = (time.perf_counter() - t0) / 5 * 1000
    
    # Quicksort
    t0 = time.perf_counter()
    for _ in range(5):
        arr = datos.copy()
        quicksort_inplace(arr)
    t_quick = (time.perf_counter() - t0) / 5 * 1000
    
    # Python sorted()
    t0 = time.perf_counter()
    for _ in range(5):
        sorted(datos)
    t_sorted = (time.perf_counter() - t0) / 5 * 1000
    
    resultados.append((n, t_merge, t_quick, t_sorted))
    print(f"n={n:>6}: Merge={t_merge:6.2f}ms  Quick={t_quick:6.2f}ms  sorted()={t_sorted:6.2f}ms  ratio={t_merge/t_quick:.2f}x")

print("\n💡 sorted() de Python usa Timsort (Merge Sort + Insertion Sort) — optimizado en C.")
print("💡 Nuestra implementación Python de Quicksort es más lenta por el overhead de Python.")
print("💡 En C/C++ Quicksort es típicamente 2x más rápido que Merge Sort.")

# Sección 8: Resumen y Próxima Clase (2 minutos)

## ✅ Lo que aprendimos hoy

1. **Partición de Lomuto**: pivot = último elemento, un puntero, más simple de implementar.
2. **Partición de Hoare**: dos punteros, ~3× menos intercambios, más eficiente.
3. **Estrategias de pivot**: aleatorio y mediana-de-3 evitan el peor caso O(n²).
4. **Complejidad**: O(n²) peor caso, O(n log n) promedio con buena elección de pivot.
5. **vs Merge Sort**: Quicksort gana en práctica (in-place, cache-friendly), Merge Sort gana en garantías (peor caso, estabilidad).

## 🔑 Regla de decisión práctica

```
¿Necesito estabilidad?  →  Merge Sort (o Timsort)
¿RAM es limitada?       →  Quicksort (in-place)
¿Datos casi ordenados?  →  Timsort o Insertion Sort
¿Rendimiento puro?      →  Introsort (Quicksort + Heapsort + Insertion)
```

## 📚 Para el Laboratorio

Implementarás:
1. Partición de Lomuto desde cero
2. Partición de Hoare desde cero
3. Quicksort con mediana-de-3
4. Comparación empírica entre variantes

> 🎙️ **[PAUSA PROFESOR]** *"¿Preguntas antes del lab?"*